# 01 — Make Raw Files Comparable

This notebook comes after:

`00_Data_Acquisition_Fixed.ipynb`

Purpose:

Convert the raw acquisition CSV files into one comparable long-format dataset.

This notebook does:

- read only approved raw CSV files;
- ignore deprecated / legacy files;
- attach variable metadata;
- standardize country/geography codes;
- flag aggregate regions;
- remove aggregate regions from the country-level output;
- create a canonical public debt variable after country-code standardization;
- export diagnostics.

This notebook does **not**:

- process WGI;
- choose the final country sample;
- keep only OECD countries;
- drop countries because of missingness;
- impute missing values;
- clip values;
- direction-align variables;
- create POSet-ready transformed variables.

WGI is excluded here because the current WGI file is a raw extracted package, not a simple `Country-Year-Value` CSV.

WGI will be processed later in:

`03_WGI_Governance_Compilation.ipynb`

Country inclusion/exclusion happens later in:

`01_Raw_Files_Coverage_Diagnostics.ipynb`

and/or:

`05_Master_Dataset_Build.ipynb`

In [1]:
# ------------------------------------------------------
# IMPORTS AND PATHS
# ------------------------------------------------------

from pathlib import Path
from datetime import datetime
import pandas as pd
import numpy as np
import re
import json
import shutil
import warnings

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)
pd.set_option("display.float_format", "{:.4f}".format)

PROJECT_ROOT = Path.cwd()

RAW_FILES_DIR = PROJECT_ROOT / "Data" / "Raw" / "Raw_Files"

PROCESSED_DIR = PROJECT_ROOT / "Data" / "Processed" / "00_Comparable_Raw_Files"
COMBINED_LONG_DIR = PROCESSED_DIR / "Combined_Long"
PER_SOURCE_DIR = PROCESSED_DIR / "Per_Source"
DERIVED_DIR = PROCESSED_DIR / "Derived"

DIAGNOSTICS_DIR = PROJECT_ROOT / "Data" / "Diagnostics" / "00_Make_Raw_Files_Comparable"
AUDIT_DIR = PROJECT_ROOT / "Data" / "Audit"

RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_TIMESTAMP = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

CLEAR_PREVIOUS_OUTPUTS = True

if CLEAR_PREVIOUS_OUTPUTS:
    for folder in [PROCESSED_DIR, DIAGNOSTICS_DIR]:
        if folder.exists():
            shutil.rmtree(folder)

for folder in [
    PROCESSED_DIR,
    COMBINED_LONG_DIR,
    PER_SOURCE_DIR,
    DERIVED_DIR,
    DIAGNOSTICS_DIR,
    AUDIT_DIR,
]:
    folder.mkdir(parents=True, exist_ok=True)

print("Run ID:", RUN_ID)
print("Project root:", PROJECT_ROOT.resolve())
print("Raw files folder:", RAW_FILES_DIR.resolve())
print("Processed folder:", PROCESSED_DIR.resolve())
print("Diagnostics folder:", DIAGNOSTICS_DIR.resolve())

Run ID: 20260622_023827
Project root: D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks
Raw files folder: D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Data\Raw\Raw_Files
Processed folder: D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Data\Processed\00_Comparable_Raw_Files
Diagnostics folder: D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Data\Diagnostics\00_Make_Raw_Files_Comparable


In [2]:
# ------------------------------------------------------
# APPROVED RAW FILE REGISTRY
# ------------------------------------------------------
# This notebook reads ONLY this registry.
# It must not loop through all CSVs in Raw_Files.
#
# WGI is intentionally excluded because WGI is a raw extracted package,
# not a Country-Year-Value CSV. WGI is processed later.

CSV_FILE_REGISTRY = [
    {
        "file_name": "OECD_RD_GDP_2000_2025.csv",
        "variable": "rd_gdp",
        "concept": "R&D expenditure as % of GDP",
        "source_family": "OECD",
        "raw_direction": "higher_better",
        "expected_unit": "percent_of_gdp",
        "is_derived": False,
        "notes": "Structural innovation capacity variable.",
    },
    {
        "file_name": "OECD_Inflation_CPI_2000_2025.csv",
        "variable": "inflation_cpi",
        "concept": "Consumer price inflation",
        "source_family": "OECD",
        "raw_direction": "higher_generally_worse",
        "expected_unit": "annual_percent_change",
        "is_derived": False,
        "notes": "Raw inflation. Later may be transformed into inflation stability.",
    },
    {
        "file_name": "OECD_Unemployment_Rate_2000_2025.csv",
        "variable": "unemployment_rate",
        "concept": "Unemployment rate",
        "source_family": "OECD",
        "raw_direction": "higher_worse",
        "expected_unit": "percent_of_labour_force",
        "is_derived": False,
        "notes": "Later may be inverted into employment strength.",
    },
    {
        "file_name": "OECD_Tertiary_Education_2000_2025.csv",
        "variable": "tertiary_education_25_64",
        "concept": "Adult tertiary educational attainment, age 25-64",
        "source_family": "OECD",
        "raw_direction": "higher_better",
        "expected_unit": "percent_of_same_age_population",
        "is_derived": False,
        "notes": "Human-capital stock variable. Not enrolment/student-flow data.",
    },
    {
        "file_name": "OECD_Productivity_GDP_per_Hour_2000_2025.csv",
        "variable": "productivity_gdp_per_hour",
        "concept": "GDP per hour worked",
        "source_family": "OECD_DBnomics",
        "raw_direction": "higher_better",
        "expected_unit": "usd_ppp_per_hour_constant_prices",
        "is_derived": False,
        "notes": "Corrected productivity variable using constant-price Q series.",
    },
    {
        "file_name": "OECD_GDP_Growth_2000_2025.csv",
        "variable": "gdp_growth",
        "concept": "Real GDP growth",
        "source_family": "OECD_DBnomics",
        "raw_direction": "context_dependent",
        "expected_unit": "annual_percent_change",
        "is_derived": False,
        "notes": "Used for recovery outcome and pre-shock GDP growth stability.",
    },
    {
        "file_name": "Eurostat_Public_Debt_GDP_2000_2025.csv",
        "variable": "public_debt_gdp_eurostat",
        "concept": "Public debt as % of GDP",
        "source_family": "Eurostat",
        "raw_direction": "higher_worse",
        "expected_unit": "percent_of_gdp",
        "is_derived": False,
        "notes": "Kept separate from World Bank debt. Used later for canonical debt.",
    },
    {
        "file_name": "WorldBank_Public_Debt_GDP_2000_2025.csv",
        "variable": "public_debt_gdp_worldbank",
        "concept": "Public debt as % of GDP",
        "source_family": "WorldBank",
        "raw_direction": "higher_worse",
        "expected_unit": "percent_of_gdp",
        "is_derived": False,
        "notes": "Kept separate from Eurostat debt. Used later for canonical debt.",
    },
    {
        "file_name": "WorldBank_Energy_Import_Dependency_Raw_2000_2025.csv",
        "variable": "energy_import_dependency_raw",
        "concept": "Energy import dependency",
        "source_family": "WorldBank",
        "raw_direction": "higher_worse",
        "expected_unit": "percent_of_energy_use_raw",
        "is_derived": False,
        "notes": "Raw values preserved. Negative values are meaningful for net exporters. Do not clip.",
    },
]

DEPRECATED_SOURCE_FILES = {
    "Global_Energy_Dependency_0_100.csv",
    "Global_Public_Debt_GDP_2000_2025.csv",
    "OECD_Productivity_Variation_2000_2025.csv",
    "Canonical_Public_Debt_GDP_2000_2025.csv",
}

registry_df = pd.DataFrame(CSV_FILE_REGISTRY)

registry_df.to_csv(
    DIAGNOSTICS_DIR / "approved_raw_file_registry.csv",
    index=False,
)

display(registry_df)

,file_name,variable,concept,source_family,raw_direction,expected_unit,is_derived,notes
0,OECD_RD_GDP_2000_2025.csv,rd_gdp,R&D expenditure as % of GDP,OECD,higher_better,percent_of_gdp,False,Structural innovation capacity variable.
1,OECD_Inflation_CPI_2000_2025.csv,inflation_cpi,Consumer price inflation,OECD,higher_generally_worse,annual_percent_change,False,Raw inflation. Later may be transformed into i...
2,OECD_Unemployment_Rate_2000_2025.csv,unemployment_rate,Unemployment rate,OECD,higher_worse,percent_of_labour_force,False,Later may be inverted into employment strength.
3,OECD_Tertiary_Education_2000_2025.csv,tertiary_education_25_64,"Adult tertiary educational attainment, age 25-64",OECD,higher_better,percent_of_same_age_population,False,Human-capital stock variable. Not enrolment/st...
4,OECD_Productivity_GDP_per_Hour_2000_2025.csv,productivity_gdp_per_hour,GDP per hour worked,OECD_DBnomics,higher_better,usd_ppp_per_hour_constant_prices,False,Corrected productivity variable using constant...
5,OECD_GDP_Growth_2000_2025.csv,gdp_growth,Real GDP growth,OECD_DBnomics,context_dependent,annual_percent_change,False,Used for recovery outcome and pre-shock GDP gr...
6,Eurostat_Public_Debt_GDP_2000_2025.csv,public_debt_gdp_eurostat,Public debt as % of GDP,Eurostat,higher_worse,percent_of_gdp,False,Kept separate from World Bank debt. Used later...
7,WorldBank_Public_Debt_GDP_2000_2025.csv,public_debt_gdp_worldbank,Public debt as % of GDP,WorldBank,higher_worse,percent_of_gdp,False,Kept separate from Eurostat debt. Used later f...
8,WorldBank_Energy_Import_Dependency_Raw_2000_20...,energy_import_dependency_raw,Energy import dependency,WorldBank,higher_worse,percent_of_energy_use_raw,False,Raw values preserved. Negative values are mean...


In [3]:
# ------------------------------------------------------
# VALUE RANGE RULES
# ------------------------------------------------------
# These are only sanity checks.
# No values are clipped or deleted here.

VALUE_RANGE_RULES = {
    "rd_gdp": {
        "expected_min": 0,
        "expected_max": 10,
        "note": "R&D as % GDP.",
    },
    "inflation_cpi": {
        "expected_min": -20,
        "expected_max": 300,
        "note": "Annual inflation percent change.",
    },
    "unemployment_rate": {
        "expected_min": 0,
        "expected_max": 50,
        "note": "Unemployment rate as % labour force.",
    },
    "tertiary_education_25_64": {
        "expected_min": 0,
        "expected_max": 100,
        "note": "Adult tertiary attainment, age 25-64.",
    },
    "productivity_gdp_per_hour": {
        "expected_min": 0,
        "expected_max": 300,
        "note": "GDP per hour worked, constant prices.",
    },
    "gdp_growth": {
        "expected_min": -30,
        "expected_max": 40,
        "note": "Annual real GDP growth.",
    },
    "public_debt_gdp_eurostat": {
        "expected_min": 0,
        "expected_max": 300,
        "note": "Public debt % GDP.",
    },
    "public_debt_gdp_worldbank": {
        "expected_min": 0,
        "expected_max": 300,
        "note": "Public debt % GDP.",
    },
    "public_debt_gdp_canonical": {
        "expected_min": 0,
        "expected_max": 300,
        "note": "Canonical public debt % GDP after standardization.",
    },
    "energy_import_dependency_raw": {
        "expected_min": -5000,
        "expected_max": 1000,
        "note": "Raw net energy imports % energy use. Negative values can be meaningful.",
    },
}

pd.DataFrame([
    {"variable": k, **v}
    for k, v in VALUE_RANGE_RULES.items()
])

,variable,expected_min,expected_max,note
0,rd_gdp,0,10,R&D as % GDP.
1,inflation_cpi,-20,300,Annual inflation percent change.
2,unemployment_rate,0,50,Unemployment rate as % labour force.
3,tertiary_education_25_64,0,100,"Adult tertiary attainment, age 25-64."
4,productivity_gdp_per_hour,0,300,"GDP per hour worked, constant prices."
5,gdp_growth,-30,40,Annual real GDP growth.
6,public_debt_gdp_eurostat,0,300,Public debt % GDP.
7,public_debt_gdp_worldbank,0,300,Public debt % GDP.
8,public_debt_gdp_canonical,0,300,Canonical public debt % GDP after standardizat...
9,energy_import_dependency_raw,-5000,1000,Raw net energy imports % energy use. Negative ...


In [4]:
# ------------------------------------------------------
# COUNTRY CODE RESOURCES
# ------------------------------------------------------

try:
    import pycountry
    PYCOUNTRY_AVAILABLE = True
except Exception:
    PYCOUNTRY_AVAILABLE = False

print("pycountry available:", PYCOUNTRY_AVAILABLE)

if PYCOUNTRY_AVAILABLE:
    VALID_ISO3_CODES = {
        country.alpha_3.upper()
        for country in pycountry.countries
        if hasattr(country, "alpha_3")
    }

    ISO2_TO_ISO3_FROM_PYCOUNTRY = {
        country.alpha_2.upper(): country.alpha_3.upper()
        for country in pycountry.countries
        if hasattr(country, "alpha_2") and hasattr(country, "alpha_3")
    }

    ISO3_TO_NAME = {
        country.alpha_3.upper(): country.name
        for country in pycountry.countries
        if hasattr(country, "alpha_3")
    }
else:
    VALID_ISO3_CODES = set()
    ISO2_TO_ISO3_FROM_PYCOUNTRY = {}
    ISO3_TO_NAME = {}

SPECIAL_CODE_MAP = {
    "EL": "GRC",
    "UK": "GBR",
    "XK": "XKX",
    "XKX": "XKX",
}

EUROSTAT_ISO2_TO_ISO3_FALLBACK = {
    "AT": "AUT", "BE": "BEL", "BG": "BGR", "CH": "CHE", "CY": "CYP",
    "CZ": "CZE", "DE": "DEU", "DK": "DNK", "EE": "EST", "EL": "GRC",
    "ES": "ESP", "FI": "FIN", "FR": "FRA", "HR": "HRV", "HU": "HUN",
    "IE": "IRL", "IS": "ISL", "IT": "ITA", "LI": "LIE", "LT": "LTU",
    "LU": "LUX", "LV": "LVA", "MT": "MLT", "NL": "NLD", "NO": "NOR",
    "PL": "POL", "PT": "PRT", "RO": "ROU", "RS": "SRB", "SE": "SWE",
    "SI": "SVN", "SK": "SVK", "TR": "TUR", "UK": "GBR",
}

ISO2_TO_ISO3 = {}
ISO2_TO_ISO3.update(ISO2_TO_ISO3_FROM_PYCOUNTRY)
ISO2_TO_ISO3.update(EUROSTAT_ISO2_TO_ISO3_FALLBACK)
ISO2_TO_ISO3.update(SPECIAL_CODE_MAP)

AGGREGATE_REGION_CODES = {
    # Eurostat / EU aggregates
    "EU", "EU15", "EU25", "EU27", "EU27_2007", "EU27_2020", "EU28",
    "EA", "EA12", "EA17", "EA18", "EA19", "EA20", "EA21",
    "EFTA", "EEA30_2007", "EEA31", "EEA32",

    # OECD / DBnomics / groups
    "OECD", "OAVG", "OECDE", "OECD_REP",
    "EU19OECD", "EU22OECD",
    "WLD", "WORLD", "G7", "G7M", "G20", "G-7", "G-20",

    # World Bank aggregates
    "ARB", "CSS", "CEB", "EAS", "EAP", "EAR", "ECS", "ECA", "TEC",
    "TEA",
    "EMU", "EUU", "FCS", "HPC", "HIC", "IBD", "IBT", "IDA", "IDX",
    "IDB", "LDC", "LCN", "LAC", "TLA", "LMY", "LIC", "LMC", "LTE",
    "MEA", "MNA", "TMN", "MIC", "NAC", "OED", "OSS", "PRE", "PST",
    "PSS", "SST", "SAS", "TSA", "SSF", "SSA", "TSS", "UMC",
    "AFE", "AFW", "CHI", "INX",
}

def clean_country_code(raw_code):
    if pd.isna(raw_code):
        return ""

    code = str(raw_code).strip().upper()
    code = code.replace(" ", "")
    return code

pycountry available: True


In [5]:
# ------------------------------------------------------
# COUNTRY STANDARDIZATION FUNCTION
# ------------------------------------------------------

def standardize_country_code(raw_code):
    raw = clean_country_code(raw_code)

    if raw == "":
        return {
            "Country_Raw": raw_code,
            "Country": np.nan,
            "country_name": np.nan,
            "standardization_status": "missing_code",
            "is_aggregate_region": False,
        }

    if raw in AGGREGATE_REGION_CODES:
        return {
            "Country_Raw": raw,
            "Country": np.nan,
            "country_name": raw,
            "standardization_status": "aggregate_region",
            "is_aggregate_region": True,
        }

    if raw in SPECIAL_CODE_MAP:
        iso3 = SPECIAL_CODE_MAP[raw]
        return {
            "Country_Raw": raw,
            "Country": iso3,
            "country_name": ISO3_TO_NAME.get(iso3, iso3),
            "standardization_status": "mapped_special_code",
            "is_aggregate_region": False,
        }

    if raw in ISO2_TO_ISO3:
        iso3 = ISO2_TO_ISO3[raw]
        return {
            "Country_Raw": raw,
            "Country": iso3,
            "country_name": ISO3_TO_NAME.get(iso3, iso3),
            "standardization_status": "mapped_iso2_to_iso3",
            "is_aggregate_region": False,
        }

    if len(raw) == 3:
        if PYCOUNTRY_AVAILABLE:
            if raw in VALID_ISO3_CODES:
                return {
                    "Country_Raw": raw,
                    "Country": raw,
                    "country_name": ISO3_TO_NAME.get(raw, raw),
                    "standardization_status": "already_iso3",
                    "is_aggregate_region": False,
                }

            return {
                "Country_Raw": raw,
                "Country": np.nan,
                "country_name": np.nan,
                "standardization_status": "unmapped_three_letter_code",
                "is_aggregate_region": False,
            }

        return {
            "Country_Raw": raw,
            "Country": raw,
            "country_name": raw,
            "standardization_status": "assumed_iso3_pycountry_unavailable",
            "is_aggregate_region": False,
        }

    return {
        "Country_Raw": raw,
        "Country": np.nan,
        "country_name": np.nan,
        "standardization_status": "unmapped_code",
        "is_aggregate_region": False,
    }


pd.DataFrame([
    standardize_country_code(x)
    for x in ["ITA", "DEU", "IT", "DE", "EL", "UK", "EU27_2020", "WLD", "ARB", "XK"]
])

,Country_Raw,Country,country_name,standardization_status,is_aggregate_region
0,ITA,ITA,Italy,already_iso3,False
1,DEU,DEU,Germany,already_iso3,False
2,IT,ITA,Italy,mapped_iso2_to_iso3,False
3,DE,DEU,Germany,mapped_iso2_to_iso3,False
4,EL,GRC,Greece,mapped_special_code,False
5,UK,GBR,United Kingdom,mapped_special_code,False
6,EU27_2020,NaN,EU27_2020,aggregate_region,True
7,WLD,NaN,WLD,aggregate_region,True
8,ARB,NaN,ARB,aggregate_region,True
9,XK,XKX,XKX,mapped_special_code,False


In [6]:
# ------------------------------------------------------
# RAW FILE LOADER
# ------------------------------------------------------

def read_and_standardize_raw_file(file_meta):
    file_name = file_meta["file_name"]
    variable = file_meta["variable"]
    path = RAW_FILES_DIR / file_name

    if not path.exists():
        raise FileNotFoundError(f"Missing raw file: {path}")

    df = pd.read_csv(path)

    required_cols = {"Country", "Year", "Value"}
    missing_cols = required_cols - set(df.columns)

    if missing_cols:
        raise ValueError(f"{file_name}: missing required columns {missing_cols}")

    out = df.copy()

    out["source_file"] = file_name
    out["variable"] = variable
    out["concept"] = file_meta["concept"]
    out["source_family"] = file_meta["source_family"]
    out["raw_direction"] = file_meta["raw_direction"]
    out["expected_unit"] = file_meta["expected_unit"]
    out["is_derived"] = file_meta["is_derived"]
    out["variable_notes"] = file_meta["notes"]

    out["Country_Raw"] = out["Country"].astype(str).str.strip().str.upper()
    out["Year"] = pd.to_numeric(out["Year"], errors="coerce")
    out["Value"] = pd.to_numeric(out["Value"], errors="coerce")

    out = out.dropna(subset=["Country_Raw", "Year", "Value"]).copy()
    out["Year"] = out["Year"].astype(int)

    standardization = pd.DataFrame([
        standardize_country_code(code)
        for code in out["Country_Raw"]
    ])

    out = pd.concat(
        [
            out.drop(columns=["Country"]),
            standardization[["Country", "country_name", "standardization_status", "is_aggregate_region"]],
        ],
        axis=1,
    )

    out = out[
        [
            "Country",
            "Country_Raw",
            "country_name",
            "Year",
            "variable",
            "Value",
            "source_file",
            "source_family",
            "concept",
            "expected_unit",
            "raw_direction",
            "is_derived",
            "variable_notes",
            "standardization_status",
            "is_aggregate_region",
        ]
    ].copy()

    return out

In [7]:
# ------------------------------------------------------
# READ ALL APPROVED RAW FILES
# ------------------------------------------------------

frames = []
file_diagnostics_rows = []
missing_files = []

for meta in CSV_FILE_REGISTRY:
    file_name = meta["file_name"]
    variable = meta["variable"]
    path = RAW_FILES_DIR / file_name

    print("=" * 80)
    print("Reading:", file_name)

    if not path.exists():
        print("  -> MISSING")
        missing_files.append(file_name)

        file_diagnostics_rows.append({
            "file_name": file_name,
            "variable": variable,
            "status": "missing",
            "rows": 0,
            "countries_raw": 0,
            "countries_standardized": 0,
            "aggregate_rows": 0,
            "unmapped_rows": 0,
            "min_year": np.nan,
            "max_year": np.nan,
        })
        continue

    df_std = read_and_standardize_raw_file(meta)

    frames.append(df_std)

    file_diagnostics_rows.append({
        "file_name": file_name,
        "variable": variable,
        "status": "read_successfully",
        "rows": len(df_std),
        "countries_raw": df_std["Country_Raw"].nunique(),
        "countries_standardized": df_std["Country"].nunique(),
        "aggregate_rows": int(df_std["is_aggregate_region"].sum()),
        "unmapped_rows": int(df_std["Country"].isna().sum() - df_std["is_aggregate_region"].sum()),
        "min_year": df_std["Year"].min(),
        "max_year": df_std["Year"].max(),
    })

    print(f"  -> rows: {len(df_std)}")
    print(f"  -> raw countries/geographies: {df_std['Country_Raw'].nunique()}")
    print(f"  -> standardized countries: {df_std['Country'].nunique()}")
    print(f"  -> year range: {df_std['Year'].min()}–{df_std['Year'].max()}")

if not frames:
    raise RuntimeError("No approved raw files were read. Check RAW_FILES_DIR and CSV_FILE_REGISTRY.")

all_long_including_aggregates = pd.concat(frames, ignore_index=True)

file_diagnostics = pd.DataFrame(file_diagnostics_rows)

file_diagnostics.to_csv(
    DIAGNOSTICS_DIR / "csv_standardization_diagnostics.csv",
    index=False,
)

display(file_diagnostics)

Reading: OECD_RD_GDP_2000_2025.csv
  -> rows: 1127
  -> raw countries/geographies: 49
  -> standardized countries: 47
  -> year range: 2000–2025
Reading: OECD_Inflation_CPI_2000_2025.csv
  -> rows: 1362
  -> raw countries/geographies: 54
  -> standardized countries: 48
  -> year range: 2000–2025
Reading: OECD_Unemployment_Rate_2000_2025.csv
  -> rows: 1441
  -> raw countries/geographies: 59
  -> standardized countries: 53
  -> year range: 2000–2025
Reading: OECD_Tertiary_Education_2000_2025.csv
  -> rows: 1003
  -> raw countries/geographies: 51
  -> standardized countries: 48
  -> year range: 2000–2024
Reading: OECD_Productivity_GDP_per_Hour_2000_2025.csv
  -> rows: 1077
  -> raw countries/geographies: 44
  -> standardized countries: 41
  -> year range: 2000–2024
Reading: OECD_GDP_Growth_2000_2025.csv
  -> rows: 1209
  -> raw countries/geographies: 47
  -> standardized countries: 44
  -> year range: 2000–2025
Reading: Eurostat_Public_Debt_GDP_2000_2025.csv
  -> rows: 806
  -> raw count

,file_name,variable,status,rows,countries_raw,countries_standardized,aggregate_rows,unmapped_rows,min_year,max_year
0,OECD_RD_GDP_2000_2025.csv,rd_gdp,read_successfully,1127,49,47,50,0,2000,2025
1,OECD_Inflation_CPI_2000_2025.csv,inflation_cpi,read_successfully,1362,54,48,152,0,2000,2025
2,OECD_Unemployment_Rate_2000_2025.csv,unemployment_rate,read_successfully,1441,59,53,156,0,2000,2025
3,OECD_Tertiary_Education_2000_2025.csv,tertiary_education_25_64,read_successfully,1003,51,48,30,0,2000,2024
4,OECD_Productivity_GDP_per_Hour_2000_2025.csv,productivity_gdp_per_hour,read_successfully,1077,44,41,74,0,2000,2024
5,OECD_GDP_Growth_2000_2025.csv,gdp_growth,read_successfully,1209,47,44,78,0,2000,2025
6,Eurostat_Public_Debt_GDP_2000_2025.csv,public_debt_gdp_eurostat,read_successfully,806,31,27,104,0,2000,2025
7,WorldBank_Public_Debt_GDP_2000_2025.csv,public_debt_gdp_worldbank,read_successfully,1243,96,87,112,0,2000,2024
8,WorldBank_Energy_Import_Dependency_Raw_2000_20...,energy_import_dependency_raw,read_successfully,4138,188,145,987,0,2000,2023


In [8]:
# ------------------------------------------------------
# COUNTRY CONVERSION DIAGNOSTICS
# ------------------------------------------------------

country_conversion_attempts = (
    all_long_including_aggregates[
        [
            "source_file",
            "variable",
            "Country_Raw",
            "Country",
            "country_name",
            "standardization_status",
            "is_aggregate_region",
        ]
    ]
    .drop_duplicates()
    .sort_values(["standardization_status", "Country_Raw", "source_file"])
    .reset_index(drop=True)
)

country_conversion_attempts.to_csv(
    DIAGNOSTICS_DIR / "country_conversion_attempts.csv",
    index=False,
)

country_standardization_status_summary = (
    all_long_including_aggregates
    .groupby("standardization_status")
    .agg(
        rows=("Value", "size"),
        raw_codes=("Country_Raw", "nunique"),
        standardized_countries=("Country", "nunique"),
    )
    .reset_index()
    .sort_values("rows", ascending=False)
)

country_standardization_status_summary.to_csv(
    DIAGNOSTICS_DIR / "country_standardization_status_summary.csv",
    index=False,
)

country_standardization_status_by_file = (
    all_long_including_aggregates
    .groupby(["source_file", "variable", "standardization_status"])
    .agg(
        rows=("Value", "size"),
        raw_codes=("Country_Raw", "nunique"),
        standardized_countries=("Country", "nunique"),
    )
    .reset_index()
    .sort_values(["source_file", "standardization_status"])
)

country_standardization_status_by_file.to_csv(
    DIAGNOSTICS_DIR / "country_standardization_status_by_file.csv",
    index=False,
)

dropped_aggregate_regions = (
    all_long_including_aggregates
    .query("is_aggregate_region == True")
    [["source_file", "variable", "Country_Raw", "Year", "Value"]]
    .sort_values(["source_file", "Country_Raw", "Year"])
)

dropped_aggregate_regions.to_csv(
    DIAGNOSTICS_DIR / "dropped_aggregate_regions.csv",
    index=False,
)

unmapped_country_codes = (
    all_long_including_aggregates
    .query("Country.isna() and is_aggregate_region == False")
    [["source_file", "variable", "Country_Raw", "Year", "Value", "standardization_status"]]
    .sort_values(["source_file", "Country_Raw", "Year"])
)

unmapped_country_codes.to_csv(
    DIAGNOSTICS_DIR / "unmapped_country_codes.csv",
    index=False,
)

display(country_standardization_status_summary)
display(unmapped_country_codes.head(50))

,standardization_status,rows,raw_codes,standardized_countries
1,already_iso3,10961,169,169
0,aggregate_region,1743,58,0
2,mapped_iso2_to_iso3,676,26,26
3,mapped_special_code,26,1,1


,source_file,variable,Country_Raw,Year,Value,standardization_status


In [9]:
# ------------------------------------------------------
# CREATE COUNTRY-LEVEL RAW COMPARABLE DATA
# ------------------------------------------------------
# Drop aggregate regions and unmapped codes from country-level output.
# They remain available in diagnostics.

country_level_raw_long = (
    all_long_including_aggregates
    .query("is_aggregate_region == False")
    .dropna(subset=["Country"])
    .copy()
)

country_level_raw_long = country_level_raw_long.sort_values(
    ["variable", "Country", "Year", "source_file"]
).reset_index(drop=True)

all_long_including_aggregates.to_csv(
    COMBINED_LONG_DIR / "all_comparable_sources_long_including_aggregates.csv",
    index=False,
)

country_level_raw_long.to_csv(
    COMBINED_LONG_DIR / "raw_only_country_level_long.csv",
    index=False,
)

print("Rows including aggregates:", len(all_long_including_aggregates))
print("Rows in country-level raw long:", len(country_level_raw_long))
print("Variables:", country_level_raw_long["variable"].nunique())
print("Countries:", country_level_raw_long["Country"].nunique())
print("Years:", country_level_raw_long["Year"].min(), "-", country_level_raw_long["Year"].max())

Rows including aggregates: 13406
Rows in country-level raw long: 11663
Variables: 9
Countries: 169
Years: 2000 - 2025


In [10]:
# ------------------------------------------------------
# DUPLICATE COUNTRY-YEAR-VARIABLE CHECKS
# ------------------------------------------------------

duplicate_check = (
    country_level_raw_long
    .groupby(["source_file", "variable", "Country", "Year"])
    .agg(
        row_count=("Value", "size"),
        unique_values=("Value", "nunique"),
        min_value=("Value", "min"),
        max_value=("Value", "max"),
    )
    .reset_index()
    .query("row_count > 1")
    .sort_values(["source_file", "variable", "Country", "Year"])
)

duplicate_check.to_csv(
    DIAGNOSTICS_DIR / "duplicate_country_year_variable_check.csv",
    index=False,
)

if len(duplicate_check) > 0:
    display(duplicate_check)

    conflicting_duplicates = duplicate_check.query("unique_values > 1")

    if len(conflicting_duplicates) > 0:
        conflicting_duplicates.to_csv(
            DIAGNOSTICS_DIR / "conflicting_duplicate_country_year_variable_check.csv",
            index=False,
        )

        raise ValueError(
            "Conflicting duplicates detected after standardization. "
            "Review conflicting_duplicate_country_year_variable_check.csv"
        )

    print("Only exact-value duplicates detected. Dropping exact duplicates.")

    country_level_raw_long = (
        country_level_raw_long
        .drop_duplicates(
            subset=[
                "source_file",
                "variable",
                "Country",
                "Year",
                "Value",
            ]
        )
        .reset_index(drop=True)
    )
else:
    print("No duplicate Country-Year-Variable keys detected.")

No duplicate Country-Year-Variable keys detected.


In [11]:
# ------------------------------------------------------
# CREATE CANONICAL PUBLIC DEBT AFTER STANDARDIZATION
# ------------------------------------------------------
# Rule:
# Use Eurostat where available.
# Use World Bank only as fallback.
#
# This is derived after standardization.
# It is not a raw acquisition file.

def create_canonical_public_debt(country_level_raw_long):
    debt = country_level_raw_long[
        country_level_raw_long["variable"].isin([
            "public_debt_gdp_eurostat",
            "public_debt_gdp_worldbank",
        ])
    ].copy()

    if debt.empty:
        raise ValueError("No public debt rows found for canonical debt creation.")

    debt_wide = (
        debt
        .pivot_table(
            index=["Country", "country_name", "Year"],
            columns="variable",
            values="Value",
            aggfunc="first",
        )
        .reset_index()
    )

    for col in ["public_debt_gdp_eurostat", "public_debt_gdp_worldbank"]:
        if col not in debt_wide.columns:
            debt_wide[col] = np.nan

    debt_wide["Value"] = np.where(
        debt_wide["public_debt_gdp_eurostat"].notna(),
        debt_wide["public_debt_gdp_eurostat"],
        debt_wide["public_debt_gdp_worldbank"],
    )

    debt_wide["Source_Used"] = np.where(
        debt_wide["public_debt_gdp_eurostat"].notna(),
        "Eurostat",
        np.where(
            debt_wide["public_debt_gdp_worldbank"].notna(),
            "WorldBank",
            np.nan,
        ),
    )

    canonical = debt_wide.dropna(subset=["Value"]).copy()

    canonical["Country_Raw"] = canonical["Country"]
    canonical["variable"] = "public_debt_gdp_canonical"
    canonical["source_file"] = "derived_after_standardization"
    canonical["source_family"] = "Derived"
    canonical["concept"] = "Canonical public debt as % of GDP"
    canonical["expected_unit"] = "percent_of_gdp"
    canonical["raw_direction"] = "higher_worse"
    canonical["is_derived"] = True
    canonical["variable_notes"] = "Derived after country-code standardization. Eurostat preferred; World Bank fallback."
    canonical["standardization_status"] = "derived_after_standardization"
    canonical["is_aggregate_region"] = False

    canonical = canonical[
        [
            "Country",
            "Country_Raw",
            "country_name",
            "Year",
            "variable",
            "Value",
            "source_file",
            "source_family",
            "concept",
            "expected_unit",
            "raw_direction",
            "is_derived",
            "variable_notes",
            "standardization_status",
            "is_aggregate_region",
            "Source_Used",
        ]
    ].copy()

    overlap = debt_wide[
        debt_wide["public_debt_gdp_eurostat"].notna()
        & debt_wide["public_debt_gdp_worldbank"].notna()
    ].copy()

    if not overlap.empty:
        overlap["absolute_difference"] = (
            overlap["public_debt_gdp_eurostat"]
            - overlap["public_debt_gdp_worldbank"]
        ).abs()

        overlap.to_csv(
            DIAGNOSTICS_DIR / "public_debt_source_comparison_country_year.csv",
            index=False,
        )

        overall_summary = pd.DataFrame([{
            "overlap_rows": len(overlap),
            "overlap_countries": overlap["Country"].nunique(),
            "correlation": overlap["public_debt_gdp_eurostat"].corr(overlap["public_debt_gdp_worldbank"]),
            "mean_absolute_difference": overlap["absolute_difference"].mean(),
            "median_absolute_difference": overlap["absolute_difference"].median(),
            "max_absolute_difference": overlap["absolute_difference"].max(),
        }])

        overall_summary.to_csv(
            DIAGNOSTICS_DIR / "public_debt_source_comparison_overall_summary.csv",
            index=False,
        )

        country_summary = (
            overlap
            .groupby("Country")
            .agg(
                overlap_years=("Year", "count"),
                mean_absolute_difference=("absolute_difference", "mean"),
                max_absolute_difference=("absolute_difference", "max"),
            )
            .reset_index()
            .sort_values("mean_absolute_difference", ascending=False)
        )

        country_summary.to_csv(
            DIAGNOSTICS_DIR / "public_debt_source_comparison_summary_by_country.csv",
            index=False,
        )

    source_usage = (
        canonical
        .groupby("Source_Used")
        .size()
        .reset_index(name="country_year_rows")
        .sort_values("country_year_rows", ascending=False)
    )

    source_usage.to_csv(
        DIAGNOSTICS_DIR / "public_debt_canonical_source_usage.csv",
        index=False,
    )

    canonical.to_csv(
        DERIVED_DIR / "public_debt_gdp_canonical_country_year.csv",
        index=False,
    )

    return canonical, source_usage


canonical_debt_long, public_debt_source_usage = create_canonical_public_debt(country_level_raw_long)

display(public_debt_source_usage)
display(canonical_debt_long.head())

,Source_Used,country_year_rows
1,WorldBank,1081
0,Eurostat,702


variable,Country,Country_Raw,country_name,Year,variable,Value,source_file,source_family,concept,expected_unit,raw_direction,is_derived,variable_notes,standardization_status,is_aggregate_region,Source_Used
0,ALB,ALB,Albania,2011,public_debt_gdp_canonical,69.1922,derived_after_standardization,Derived,Canonical public debt as % of GDP,percent_of_gdp,higher_worse,True,Derived after country-code standardization. Eu...,derived_after_standardization,False,WorldBank
1,ALB,ALB,Albania,2012,public_debt_gdp_canonical,64.0504,derived_after_standardization,Derived,Canonical public debt as % of GDP,percent_of_gdp,higher_worse,True,Derived after country-code standardization. Eu...,derived_after_standardization,False,WorldBank
2,ALB,ALB,Albania,2013,public_debt_gdp_canonical,70.4663,derived_after_standardization,Derived,Canonical public debt as % of GDP,percent_of_gdp,higher_worse,True,Derived after country-code standardization. Eu...,derived_after_standardization,False,WorldBank
3,ALB,ALB,Albania,2014,public_debt_gdp_canonical,72.9443,derived_after_standardization,Derived,Canonical public debt as % of GDP,percent_of_gdp,higher_worse,True,Derived after country-code standardization. Eu...,derived_after_standardization,False,WorldBank
4,ALB,ALB,Albania,2015,public_debt_gdp_canonical,79.2843,derived_after_standardization,Derived,Canonical public debt as % of GDP,percent_of_gdp,higher_worse,True,Derived after country-code standardization. Eu...,derived_after_standardization,False,WorldBank


In [12]:
# ------------------------------------------------------
# FINAL COMBINED LONG DATASET
# ------------------------------------------------------

country_level_raw_long["Source_Used"] = np.nan

all_comparable_sources_long = pd.concat(
    [
        country_level_raw_long,
        canonical_debt_long,
    ],
    ignore_index=True,
)

all_comparable_sources_long = all_comparable_sources_long.sort_values(
    ["variable", "Country", "Year"]
).reset_index(drop=True)

all_comparable_sources_long.to_csv(
    COMBINED_LONG_DIR / "all_comparable_sources_long.csv",
    index=False,
)

print("Final comparable long dataset saved:")
print(COMBINED_LONG_DIR / "all_comparable_sources_long.csv")
print()
print("Rows:", len(all_comparable_sources_long))
print("Variables:", all_comparable_sources_long["variable"].nunique())
print("Countries:", all_comparable_sources_long["Country"].nunique())
print("Years:", all_comparable_sources_long["Year"].min(), "-", all_comparable_sources_long["Year"].max())

display(
    all_comparable_sources_long
    .groupby("variable")
    .agg(
        rows=("Value", "size"),
        countries=("Country", "nunique"),
        min_year=("Year", "min"),
        max_year=("Year", "max"),
        min_value=("Value", "min"),
        max_value=("Value", "max"),
    )
    .reset_index()
    .sort_values("variable")
)

Final comparable long dataset saved:
D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Data\Processed\00_Comparable_Raw_Files\Combined_Long\all_comparable_sources_long.csv

Rows: 13446
Variables: 10
Countries: 169
Years: 2000 - 2025


,variable,rows,countries,min_year,max_year,min_value,max_value
0,energy_import_dependency_raw,3151,145,2000,2023,-3300.1473,450.1815
1,gdp_growth,1131,44,2000,2025,-16.0400,24.6240
2,inflation_cpi,1210,48,2000,2025,-4.4475,219.8845
3,productivity_gdp_per_hour,1003,41,2000,2024,11.7448,142.9044
4,public_debt_gdp_canonical,1783,112,2000,2025,-1.1707,209.4000
5,public_debt_gdp_eurostat,702,27,2000,2025,3.9000,209.4000
6,public_debt_gdp_worldbank,1131,87,2000,2024,-1.1707,194.6828
7,rd_gdp,1077,47,2000,2025,0.1401,6.9589
8,tertiary_education_25_64,973,48,2000,2024,5.4441,64.6622
9,unemployment_rate,1285,53,2000,2025,0.2510,36.0250


In [13]:
# ------------------------------------------------------
# VARIABLE INVENTORY
# ------------------------------------------------------

standardized_variable_inventory = (
    all_comparable_sources_long
    .groupby([
        "variable",
        "concept",
        "source_family",
        "expected_unit",
        "raw_direction",
        "is_derived",
    ])
    .agg(
        rows=("Value", "size"),
        countries=("Country", "nunique"),
        min_year=("Year", "min"),
        max_year=("Year", "max"),
        min_value=("Value", "min"),
        median_value=("Value", "median"),
        max_value=("Value", "max"),
    )
    .reset_index()
    .sort_values("variable")
)

standardized_variable_inventory.to_csv(
    DIAGNOSTICS_DIR / "standardized_variable_inventory.csv",
    index=False,
)

variable_metadata = (
    all_comparable_sources_long
    [[
        "variable",
        "concept",
        "source_family",
        "expected_unit",
        "raw_direction",
        "is_derived",
        "variable_notes",
    ]]
    .drop_duplicates()
    .sort_values("variable")
)

variable_metadata.to_csv(
    DIAGNOSTICS_DIR / "variable_metadata.csv",
    index=False,
)

display(standardized_variable_inventory)

,variable,concept,source_family,expected_unit,raw_direction,is_derived,rows,countries,min_year,max_year,min_value,median_value,max_value
0,energy_import_dependency_raw,Energy import dependency,WorldBank,percent_of_energy_use_raw,higher_worse,False,3151,145,2000,2023,-3300.1473,24.3911,450.1815
1,gdp_growth,Real GDP growth,OECD_DBnomics,annual_percent_change,context_dependent,False,1131,44,2000,2025,-16.0400,2.6231,24.6240
2,inflation_cpi,Consumer price inflation,OECD,annual_percent_change,higher_generally_worse,False,1210,48,2000,2025,-4.4475,2.6000,219.8845
3,productivity_gdp_per_hour,GDP per hour worked,OECD_DBnomics,usd_ppp_per_hour_constant_prices,higher_better,False,1003,41,2000,2024,11.7448,51.2618,142.9044
4,public_debt_gdp_canonical,Canonical public debt as % of GDP,Derived,percent_of_gdp,higher_worse,True,1783,112,2000,2025,-1.1707,50.3000,209.4000
5,public_debt_gdp_eurostat,Public debt as % of GDP,Eurostat,percent_of_gdp,higher_worse,False,702,27,2000,2025,3.9000,53.6500,209.4000
6,public_debt_gdp_worldbank,Public debt as % of GDP,WorldBank,percent_of_gdp,higher_worse,False,1131,87,2000,2024,-1.1707,49.9647,194.6828
7,rd_gdp,R&D expenditure as % of GDP,OECD,percent_of_gdp,higher_better,False,1077,47,2000,2025,0.1401,1.5436,6.9589
8,tertiary_education_25_64,"Adult tertiary educational attainment, age 25-64",OECD,percent_of_same_age_population,higher_better,False,973,48,2000,2024,5.4441,30.7121,64.6622
9,unemployment_rate,Unemployment rate,OECD,percent_of_labour_force,higher_worse,False,1285,53,2000,2025,0.2510,6.5590,36.0250


In [14]:
# ------------------------------------------------------
# COUNTRY-VARIABLE COVERAGE MATRICES
# ------------------------------------------------------

country_variable_presence_matrix = (
    all_comparable_sources_long
    .assign(present=1)
    .pivot_table(
        index="Country",
        columns="variable",
        values="present",
        aggfunc="max",
        fill_value=0,
    )
    .reset_index()
)

country_variable_presence_matrix.to_csv(
    DIAGNOSTICS_DIR / "country_variable_presence_matrix.csv",
    index=False,
)

country_variable_year_count_matrix = (
    all_comparable_sources_long
    .pivot_table(
        index="Country",
        columns="variable",
        values="Year",
        aggfunc="nunique",
        fill_value=0,
    )
    .reset_index()
)

country_variable_year_count_matrix.to_csv(
    DIAGNOSTICS_DIR / "country_variable_year_count_matrix.csv",
    index=False,
)

display(country_variable_presence_matrix.head())
display(country_variable_year_count_matrix.head())

variable,Country,energy_import_dependency_raw,gdp_growth,inflation_cpi,productivity_gdp_per_hour,public_debt_gdp_canonical,public_debt_gdp_eurostat,public_debt_gdp_worldbank,rd_gdp,tertiary_education_25_64,unemployment_rate
0,AGO,1,0,0,0,0,0,0,0,0,0
1,ALB,1,0,0,0,1,0,1,0,0,0
2,AND,0,0,0,0,1,0,1,0,0,0
3,ARE,1,0,0,0,1,0,1,0,0,0
4,ARG,1,0,1,0,0,0,0,1,1,1


variable,Country,energy_import_dependency_raw,gdp_growth,inflation_cpi,productivity_gdp_per_hour,public_debt_gdp_canonical,public_debt_gdp_eurostat,public_debt_gdp_worldbank,rd_gdp,tertiary_education_25_64,unemployment_rate
0,AGO,23,0,0,0,0,0,0,0,0,0
1,ALB,24,0,0,0,14,0,14,0,0,0
2,AND,0,0,0,0,7,0,7,0,0,0
3,ARE,23,0,0,0,1,0,1,0,0,0
4,ARG,24,0,8,0,0,0,0,24,17,8


In [15]:
# ------------------------------------------------------
# VALUE RANGE SANITY CHECKS
# ------------------------------------------------------

range_rows = []

for variable, rule in VALUE_RANGE_RULES.items():
    subset = all_comparable_sources_long[
        all_comparable_sources_long["variable"] == variable
    ].copy()

    if subset.empty:
        range_rows.append({
            "variable": variable,
            "status": "missing_variable",
            "rows": 0,
            "countries": 0,
            "min_year": np.nan,
            "max_year": np.nan,
            "min_value": np.nan,
            "p01": np.nan,
            "p05": np.nan,
            "median": np.nan,
            "mean": np.nan,
            "p95": np.nan,
            "p99": np.nan,
            "max_value": np.nan,
            "below_expected_min": np.nan,
            "above_expected_max": np.nan,
            "note": rule["note"],
        })
        continue

    values = pd.to_numeric(subset["Value"], errors="coerce")

    range_rows.append({
        "variable": variable,
        "status": "checked",
        "rows": len(subset),
        "countries": subset["Country"].nunique(),
        "min_year": subset["Year"].min(),
        "max_year": subset["Year"].max(),
        "min_value": values.min(),
        "p01": values.quantile(0.01),
        "p05": values.quantile(0.05),
        "median": values.median(),
        "mean": values.mean(),
        "p95": values.quantile(0.95),
        "p99": values.quantile(0.99),
        "max_value": values.max(),
        "below_expected_min": int((values < rule["expected_min"]).sum()),
        "above_expected_max": int((values > rule["expected_max"]).sum()),
        "note": rule["note"],
    })

value_range_sanity_checks = pd.DataFrame(range_rows)

value_range_sanity_checks.to_csv(
    DIAGNOSTICS_DIR / "value_range_sanity_checks.csv",
    index=False,
)

display(value_range_sanity_checks)

,variable,status,rows,countries,min_year,max_year,min_value,p01,p05,median,mean,p95,p99,max_value,below_expected_min,above_expected_max,note
0,rd_gdp,checked,1077,47,2000,2025,0.1401,0.1977,0.3803,1.5436,1.7223,3.4930,4.6660,6.9589,0,0,R&D as % GDP.
1,inflation_cpi,checked,1210,48,2000,2025,-4.4475,-1.0710,-0.1451,2.6000,4.1721,10.8153,41.2869,219.8845,0,0,Annual inflation percent change.
2,unemployment_rate,checked,1285,53,2000,2025,0.2510,1.0755,2.9802,6.5590,7.8680,17.8616,29.0277,36.0250,0,0,Unemployment rate as % labour force.
3,tertiary_education_25_64,checked,973,48,2000,2024,5.4441,8.1490,11.8745,30.7121,30.7382,50.3981,56.2264,64.6622,0,0,"Adult tertiary attainment, age 25-64."
4,productivity_gdp_per_hour,checked,1003,41,2000,2024,11.7448,14.0744,21.4136,51.2618,55.5949,90.2477,114.6753,142.9044,0,0,"GDP per hour worked, constant prices."
5,gdp_growth,checked,1131,44,2000,2025,-16.0400,-8.1661,-3.2763,2.6231,2.7123,8.2960,10.9316,24.6240,0,0,Annual real GDP growth.
6,public_debt_gdp_eurostat,checked,702,27,2000,2025,3.9000,5.6010,13.0200,53.6500,60.6201,128.3750,180.3920,209.4000,0,0,Public debt % GDP.
7,public_debt_gdp_worldbank,checked,1131,87,2000,2024,-1.1707,6.2702,14.2752,49.9647,54.4821,113.5331,151.4268,194.6828,1,0,Public debt % GDP.
8,public_debt_gdp_canonical,checked,1783,112,2000,2025,-1.1707,5.8994,13.2729,50.3000,56.1991,117.9459,160.8538,209.4000,1,0,Canonical public debt % GDP after standardizat...
9,energy_import_dependency_raw,checked,3151,145,2000,2023,-3300.1473,-896.6913,-405.1937,24.3911,-35.7192,101.3053,243.6463,450.1815,0,0,Raw net energy imports % energy use. Negative ...


In [16]:
# ------------------------------------------------------
# LEGACY FILE SAFETY CHECK
# ------------------------------------------------------

raw_files_present = {p.name for p in RAW_FILES_DIR.glob("*.csv")}
deprecated_present = sorted(raw_files_present.intersection(DEPRECATED_SOURCE_FILES))

legacy_file_check = pd.DataFrame({
    "deprecated_file_present_in_raw_folder": deprecated_present
})

legacy_file_check.to_csv(
    DIAGNOSTICS_DIR / "deprecated_files_present_but_ignored.csv",
    index=False,
)

print("Deprecated files present but ignored:", deprecated_present)

variables_present = sorted(all_comparable_sources_long["variable"].unique())

legacy_variable_names = {
    "energy_dependency_global",
    "public_debt_gdp_global",
    "productivity_variation",
}

legacy_variables_found = sorted(set(variables_present).intersection(legacy_variable_names))

if legacy_variables_found:
    raise ValueError(f"Legacy variables found in final dataset: {legacy_variables_found}")

print("No legacy variables found in final comparable dataset.")

Deprecated files present but ignored: []
No legacy variables found in final comparable dataset.


In [17]:
# ------------------------------------------------------
# CREATE STEP 00 AUDIT WORKBOOK
# ------------------------------------------------------

STEP00_AUDIT_XLSX = AUDIT_DIR / "00_Make_Raw_Files_Comparable_Audit.xlsx"

audit_sources = [
    {
        "sheet_name": "01_file_diagnostics",
        "description": "Diagnostics for approved raw files read by this notebook.",
        "path": DIAGNOSTICS_DIR / "csv_standardization_diagnostics.csv",
    },
    {
        "sheet_name": "02_var_inventory",
        "description": "Standardized variable inventory.",
        "path": DIAGNOSTICS_DIR / "standardized_variable_inventory.csv",
    },
    {
        "sheet_name": "03_var_metadata",
        "description": "Variable metadata.",
        "path": DIAGNOSTICS_DIR / "variable_metadata.csv",
    },
    {
        "sheet_name": "04_country_status",
        "description": "Country standardization status summary.",
        "path": DIAGNOSTICS_DIR / "country_standardization_status_summary.csv",
    },
    {
        "sheet_name": "05_country_by_file",
        "description": "Country standardization status by file.",
        "path": DIAGNOSTICS_DIR / "country_standardization_status_by_file.csv",
    },
    {
        "sheet_name": "06_unmapped_codes",
        "description": "Unmapped non-aggregate country/geography codes.",
        "path": DIAGNOSTICS_DIR / "unmapped_country_codes.csv",
    },
    {
        "sheet_name": "07_aggregates",
        "description": "Aggregate region rows excluded from country-level output.",
        "path": DIAGNOSTICS_DIR / "dropped_aggregate_regions.csv",
    },
    {
        "sheet_name": "08_duplicates",
        "description": "Duplicate Country-Year-Variable checks.",
        "path": DIAGNOSTICS_DIR / "duplicate_country_year_variable_check.csv",
    },
    {
        "sheet_name": "09_value_ranges",
        "description": "Value range sanity checks.",
        "path": DIAGNOSTICS_DIR / "value_range_sanity_checks.csv",
    },
    {
        "sheet_name": "10_debt_usage",
        "description": "Canonical public debt source usage.",
        "path": DIAGNOSTICS_DIR / "public_debt_canonical_source_usage.csv",
    },
]


def safe_sheet_name(name, used):
    cleaned = re.sub(r"[/\\*\[\]:?]", "_", str(name))[:31]
    base = cleaned
    i = 1

    while cleaned in used:
        suffix = f"_{i}"
        cleaned = base[:31 - len(suffix)] + suffix
        i += 1

    used.add(cleaned)
    return cleaned


def estimate_width(series, col_name, min_width=10, max_width=60):
    values = [str(col_name)] + series.dropna().astype(str).head(200).tolist()

    if not values:
        return min_width

    return max(min_width, min(max(len(v) for v in values) + 2, max_width))


used_sheets = set()

with pd.ExcelWriter(STEP00_AUDIT_XLSX, engine="xlsxwriter") as writer:
    workbook = writer.book

    title_fmt = workbook.add_format({
        "bold": True,
        "font_size": 15,
        "font_color": "white",
        "bg_color": "#1F4E78",
        "align": "left",
        "valign": "vcenter",
    })

    desc_fmt = workbook.add_format({
        "text_wrap": True,
        "font_size": 10,
        "font_color": "#333333",
        "valign": "top",
    })

    header_fmt = workbook.add_format({
        "bold": True,
        "font_color": "white",
        "bg_color": "#5B9BD5",
        "border": 1,
        "align": "center",
        "valign": "vcenter",
        "text_wrap": True,
    })

    index_rows = []

    for item in audit_sources:
        path = Path(item["path"])

        if path.exists():
            try:
                df_temp = pd.read_csv(path)
                rows = len(df_temp)
                cols = len(df_temp.columns)
            except Exception:
                rows = np.nan
                cols = np.nan
        else:
            rows = 0
            cols = 0

        index_rows.append({
            "Sheet": item["sheet_name"],
            "Rows": rows,
            "Columns": cols,
            "Description": item["description"],
            "Path": str(path),
        })

    index_df = pd.DataFrame(index_rows)
    index_df.to_excel(writer, sheet_name="00_INDEX", index=False, startrow=5)

    ws = writer.sheets["00_INDEX"]
    ws.merge_range(0, 0, 0, max(4, len(index_df.columns) - 1), "01 Make Raw Files Comparable Audit", title_fmt)
    ws.merge_range(1, 0, 3, max(4, len(index_df.columns) - 1), "Audit workbook for Step 00 standardization and comparability checks.", desc_fmt)

    for col_idx, col_name in enumerate(index_df.columns):
        ws.write(5, col_idx, col_name, header_fmt)
        ws.set_column(col_idx, col_idx, estimate_width(index_df[col_name], col_name))

    ws.autofilter(5, 0, 5 + len(index_df), len(index_df.columns) - 1)
    ws.freeze_panes(6, 0)

    for item in audit_sources:
        path = Path(item["path"])

        if path.exists():
            try:
                df_sheet = pd.read_csv(path)
            except Exception as e:
                df_sheet = pd.DataFrame({"message": [f"Could not read file: {e}"]})
        else:
            df_sheet = pd.DataFrame({"message": ["File not found."]})

        if df_sheet.empty:
            df_sheet = pd.DataFrame({"message": ["No rows in this table."]})

        sheet_name = safe_sheet_name(item["sheet_name"], used_sheets)

        df_sheet.to_excel(writer, sheet_name=sheet_name, index=False, startrow=4)

        ws = writer.sheets[sheet_name]
        max_col = max(4, len(df_sheet.columns) - 1)

        ws.merge_range(0, 0, 0, max_col, item["sheet_name"], title_fmt)
        ws.merge_range(1, 0, 3, max_col, item["description"], desc_fmt)

        for col_idx, col_name in enumerate(df_sheet.columns):
            ws.write(4, col_idx, col_name, header_fmt)
            ws.set_column(col_idx, col_idx, estimate_width(df_sheet[col_name], col_name))

        ws.autofilter(4, 0, 4 + len(df_sheet), len(df_sheet.columns) - 1)
        ws.freeze_panes(5, 0)

print("Step 00 audit workbook created:")
print(STEP00_AUDIT_XLSX.resolve())

Step 00 audit workbook created:
D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Data\Audit\00_Make_Raw_Files_Comparable_Audit.xlsx


In [18]:
# ------------------------------------------------------
# FINAL COMPLETION CHECK
# ------------------------------------------------------

print("01 MAKE RAW FILES COMPARABLE COMPLETE")
print("=" * 80)

print("\nMain output:")
print(COMBINED_LONG_DIR / "all_comparable_sources_long.csv")

print("\nOther outputs:")
print(COMBINED_LONG_DIR / "raw_only_country_level_long.csv")
print(COMBINED_LONG_DIR / "all_comparable_sources_long_including_aggregates.csv")
print(DERIVED_DIR / "public_debt_gdp_canonical_country_year.csv")

print("\nDiagnostics folder:")
print(DIAGNOSTICS_DIR.resolve())

print("\nAudit workbook:")
print(STEP00_AUDIT_XLSX.resolve())

print("\nFinal dataset summary:")
print("Rows:", len(all_comparable_sources_long))
print("Variables:", all_comparable_sources_long["variable"].nunique())
print("Countries:", all_comparable_sources_long["Country"].nunique())
print("Year range:", all_comparable_sources_long["Year"].min(), "-", all_comparable_sources_long["Year"].max())

print("\nVariables included:")
for variable in sorted(all_comparable_sources_long["variable"].unique()):
    print("-", variable)

print("\nImportant notes:")
print("1. No final country sample selection was done here.")
print("2. WGI was not processed here. It remains for 03_WGI_Governance_Compilation.ipynb.")
print("3. Aggregates were excluded from the country-level output and saved in diagnostics.")
print("4. Canonical public debt was created after country-code standardization.")
print("5. Energy import dependency remains raw and unclipped.")
print("6. No imputation, direction alignment, or POSet transformation was done.")

print("\nNext notebook:")
print("02_Raw_Files_Coverage_Diagnostics.ipynb")

00 MAKE RAW FILES COMPARABLE COMPLETE

Main output:
D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Data\Processed\00_Comparable_Raw_Files\Combined_Long\all_comparable_sources_long.csv

Other outputs:
D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Data\Processed\00_Comparable_Raw_Files\Combined_Long\raw_only_country_level_long.csv
D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Data\Processed\00_Comparable_Raw_Files\Combined_Long\all_comparable_sources_long_including_aggregates.csv
D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Data\Processed\00_Comparable_Raw_Files\Derived\public_debt_gdp_canonical_country_year.csv

Diagnostics folder:
D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Proje